In [23]:
def execute_dag(tasks: dict[str, dict] | None) -> dict[str, int]:
    if not tasks:
        return {}

    res = {}

    def compute(name: str) -> int:
        if name not in res:
            task = tasks[name]
            dep_sum = sum(compute(d) for d in task.get("deps", []))
            op, val = task["op"].split()
            res[name] = dep_sum + int(val) if op == "add" else dep_sum * int(val)
        return res[name]

    for name in tasks:
        print(name)
        compute(name)

    return res


In [24]:

data={"load": {"op":"add 0","deps":["clean","enrich"]},
  "clean": {"op":"add 1","deps":["extract"]},
  "enrich": {"op":"mul 2","deps":["extract"]},
  "extract": {"op":"add 10","deps":[]}
}

In [25]:
data

{'load': {'op': 'add 0', 'deps': ['clean', 'enrich']},
 'clean': {'op': 'add 1', 'deps': ['extract']},
 'enrich': {'op': 'mul 2', 'deps': ['extract']},
 'extract': {'op': 'add 10', 'deps': []}}

In [26]:
print(execute_dag(data))

load
clean
enrich
extract
{'extract': 10, 'clean': 11, 'enrich': 20, 'load': 31}


In [28]:
def execute_dag(tasks: dict[str, dict] | None) -> dict[str, int]:
    # Handle None or empty dictionary inputs
    if not tasks:
        return {}

    result = {}

    def evaluate(task_name: str) -> int:
        # If task has already been calculated, return cached result
        if task_name in result:
            return result[task_name]

        task_info = tasks[task_name]
        deps = task_info.get("deps", [])
        op_str = task_info.get("op", "add 0")

        # 1. Combine outputs of all dependencies (sum to 0 if deps is empty)
        combined_val = sum(evaluate(dep) for dep in deps)

        # 2. Parse operation string e.g., "add 3" or "mul 2"
        action, num_str = op_str.split()
        num = int(num_str)

        # 3. Apply operation to combined dependency output
        if action == "add":
            task_result = combined_val + num
        elif action == "mul":
            task_result = combined_val * num
        elif action == "sub":
            task_result = combined_val - num
        elif action == "div":
            task_result = combined_val // num
        else:
            task_result = combined_val

        # Cache and return result
        result[task_name] = task_result
        return task_result

    # Evaluate every task to ensure complete output dictionary
    for name in tasks:
        evaluate(name)

    return result


In [30]:
print(execute_dag(data))


{'extract': 10, 'clean': 11, 'enrich': 20, 'load': 31}


In [31]:
def execute_dag(tasks: dict[str, dict] | None) -> dict[str, int]:
    if not tasks:
        return {}

    res = {}

    def compute(name: str) -> int:
        if name not in res:
            # 1. Sum up dependency outputs
            dep_sum = sum(compute(dep) for dep in tasks[name].get("deps", []))

            # 2. Parse operation ("add X" or "mul X")
            op, val = tasks[name]["op"].split()

            # 3. Apply operation and store in result dictionary
            res[name] = dep_sum + int(val) if op == "add" else dep_sum * int(val)

        return res[name]

    # Evaluate all tasks into the result dictionary
    return {name: compute(name) for name in tasks}


In [32]:
print(execute_dag(data))


{'load': 31, 'clean': 11, 'enrich': 20, 'extract': 10}


In [35]:
import asyncio

def execute_dag(tasks: dict[str, dict] | None) -> dict[str, int]:
    if not tasks:
        return {}

    async def main():
        res = {}
        events = {k: asyncio.Event() for k in tasks}

        async def run_task(name):
            task = tasks[name]
            deps = task.get("deps", [])
            for dep in deps:
                await events[dep].wait()

            dep_sum = sum(res[d] for d in deps)
            op, val = task["op"].split()
            res[name] = dep_sum + int(val) if op == "add" else dep_sum * int(val)
            events[name].set()

        await asyncio.gather(*(run_task(k) for k in tasks))
        return res

    return asyncio.run(main())


In [36]:
print(execute_dag(data))


RuntimeError: asyncio.run() cannot be called from a running event loop